In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install --upgrade torch-geometric-signed-directed networkx

In [0]:
dbutils.library.restartPython()

In [0]:
import shutil
import os, json

deleted = []

# RESET_GNN_TRAINING = False  # Set True to clear checkpoints/logs and restart GNN training from scratch
RESET_GNN_TRAINING = True  # Set True to clear checkpoints/logs and restart GNN training from scratch

# N_USERS = 300  # Total users to sample (proportionally across groups). None = all users.
N_USERS = None  # Total users to sample (proportionally across groups). None = all users.
EXPERIMENT_TAG = "v3_full"  # Experiment identifier

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}" if N_USERS else EXPERIMENT_TAG
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
# EXPERIMENT_DIR = f"/local_disk0/experiments/{FINAL_TAG}"

# --- RESET_GNN_TRAINING: clear checkpoints and logs (forces training restart) ---
if RESET_GNN_TRAINING:
    print("⚠️  RESET_GNN_TRAINING=True — clearing checkpoints and training logs...")
    _training_artifacts = [
        "best_retweet_gnn_general.pt",
        "training.log",
        "training_history.json",
        "training_history.pkl",
        "checkpoint.pt",
        "training_checkpoint.pt",
        "train_eval_metadata.json",
        "training_config.json",
    ]
    for fname in _training_artifacts:
        fpath = os.path.join(EXPERIMENT_DIR, fname)
        if os.path.exists(fpath):
            os.remove(fpath)
            deleted.append(fpath)

    # Also remove new-format train-eval metadata files (train_eval_metadata_*.json)
    import glob
    for _meta_file in glob.glob(os.path.join(EXPERIMENT_DIR, "train_eval_metadata_*.json")):
        os.remove(_meta_file)
        deleted.append(_meta_file)

    # Also remove legacy disk-cached train-eval samples if present
    _legacy_cache = os.path.join(EXPERIMENT_DIR, "train_eval_samples_cache")
    if os.path.isdir(_legacy_cache):
        shutil.rmtree(_legacy_cache)
        deleted.append(_legacy_cache)

    print(f"  Cleared training artifacts. GNN training will start fresh.")
else:
    print("RESET_GNN_TRAINING=False — resuming from existing checkpoint if available.")

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

if deleted:
    print(f"\nTotal deleted items: {len(deleted)}")
    for d in deleted:
        print(f"  - {d}")

In [0]:

# ============================================================
# Parameters
# ============================================================
HPARAMS_FILE = "hparams_from_gpu_tuning.json"  # Tuned hparams JSON (None = use defaults below)

# Paths
DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
# DATA_PATH = "/serafin/pcelayes/repos/sna_classifier/data"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"


os.makedirs(EXPERIMENT_DIR, exist_ok=True)
# Training hyperparameters
TRAIN_CHUNK_SIZE = 10  # Users per chunk for on-the-fly GNN sample generation during training
BATCH_SIZE = 128  # Samples per mini-batch
DROPOUT = 0.1  # Model dropout rate
DROP_EDGE_RATE = 0.1  # Edge dropout rate in GNN

# EPOCHS = 60  # Total training epochs.
# LOG_EVERY_N_STEPS = 300  # Log metrics (loss, val F1) every N training steps.
# TRAIN_F1_EVERY_N_EPOCHS = 5  # Compute train F1 every N epochs. None = skip during training (always computed at end on best model).
# EPOCHS = 10  # Total training epochs.
EPOCHS = 1  # Total training epochs.
LOG_EVERY_N_STEPS = 2000  # Log metrics (loss, val F1) every N training steps.
TRAIN_F1_EVERY_N_EPOCHS = 1  # Compute train F1 every N epochs. None = skip during training (always computed at end on best model).

PATIENCE_STEPS = 10_000  # Early stopping: steps without val F1 improvement. None = disabled.
PATIENCE = -(-PATIENCE_STEPS // LOG_EVERY_N_STEPS) if PATIENCE_STEPS else None  # Convert to checkpoints (rounded up)
GRADIENT_ACCUMULATION_STEPS = 8  # None or 1 to disable. Effective batch = batch_size * this.
MIXED_PRECISION = False  # Keep full precision; this run produced non-finite training loss under float16 autocast.
TARGET_POS_RATE = 0.10  # Oversample positives in train loader to reach this rate. None = disabled.
# MAX_VAL_SAMPLES = 50_000  # Cap validation samples (randomly sampled from test set). None = use all.
# MAX_TRAIN_EVAL_SAMPLES = 80_000  # Cap train-eval samples for checkpoint F1/loss computations. None = use all.
MAX_VAL_SAMPLES = 20_000  # Cap validation samples (randomly sampled from test set). None = use all.
MAX_TRAIN_EVAL_SAMPLES = 20_000  # Cap train-eval samples for checkpoint F1/loss computations. None = use all.

# Pre-trained weights initialization (set to a .pt file path to warm-start from another run)
# Example: "./experiments/v3_full_N150/best_retweet_gnn_general.pt"
INIT_WEIGHTS_PATH = "./experiments/v2_N20/best_retweet_gnn_general.pt"  # None = train from scratch
# INIT_WEIGHTS_PATH = None
FINETUNE_LR_FACTOR = 0.05  # When fine-tuning (INIT_WEIGHTS_PATH set), multiply base LR by this factor
FINETUNE_WD_FACTOR = 0.2  # When fine-tuning, multiply base weight_decay by this factor (less L2 → preserve pre-trained structure)
FINETUNE_WARMUP_EPOCHS = 2  # Shorter warmup when fine-tuning (weights already in a good region)

# --- Override from tuned hparams file ---
_TUNED_LR = None
_TUNED_WD = None
if HPARAMS_FILE:
    with open(HPARAMS_FILE) as _f:
        _tuned = json.load(_f)
    BATCH_SIZE = _tuned.get("batch_size", BATCH_SIZE)
    TRAIN_CHUNK_SIZE = _tuned.get("train_chunk_size", TRAIN_CHUNK_SIZE)
    GRADIENT_ACCUMULATION_STEPS = _tuned.get("grad_accum_steps", GRADIENT_ACCUMULATION_STEPS)
    DROPOUT = _tuned.get("dropout", DROPOUT)
    DROP_EDGE_RATE = _tuned.get("drop_edge_rate", DROP_EDGE_RATE)
    FINETUNE_WARMUP_EPOCHS = _tuned.get("warmup_epochs", FINETUNE_WARMUP_EPOCHS)
    _TUNED_LR = _tuned.get("lr")
    _TUNED_WD = _tuned.get("weight_decay")
    print(f"Loaded tuned hparams from: {HPARAMS_FILE}")

print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")
if INIT_WEIGHTS_PATH:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
else:
    print("Training from scratch.")

In [0]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow C++ info/warning logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # Suppress oneDNN messages

import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

In [0]:
import torch
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {props.name}, free={free/1e9:.1f}GB / {total/1e9:.1f}GB")

In [0]:
import subprocess

def get_free_memory_per_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    return [int(x) for x in result.stdout.strip().split("\n")]

def get_best_gpu():
    free_mem = get_free_memory_per_gpu()
    best_gpu = max(range(len(free_mem)), key=lambda i: free_mem[i])
    print(f"Free memory per GPU (MiB): {free_mem}")
    print(f"Selected GPU {best_gpu} with {free_mem[best_gpu]} MiB free")
    return best_gpu

device = torch.device(f"cuda:{get_best_gpu()}" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [0]:
# Load user splits
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

print(f"Groups in user_splits: {list(user_splits.keys())}")
for k, v in user_splits.items():
    print(f"  {k}: {len(v)} users")

# Define train groups vs test groups
TRAIN_GROUPS = ["u_train", "au_train"]
TEST_GROUPS = [g for g in user_splits.keys() if g not in TRAIN_GROUPS]
print(f"\nTrain groups: {TRAIN_GROUPS}")
print(f"Test groups: {TEST_GROUPS}")

# ---------------------------------------------------------------
# Deterministic sample tied to FINAL_TAG: save/load sampled user IDs
# so that re-runs for the same experiment tag use the exact same users.
# ---------------------------------------------------------------
SAMPLE_PATH = f"{EXPERIMENT_DIR}/user_sample.json"

if os.path.exists(SAMPLE_PATH):
    # --- FAST PATH: load previously saved sample for this tag ---
    with open(SAMPLE_PATH) as f:
        saved_sample = json.load(f)  # {group: [uid, uid, ...]}
    print(f"\nLoading saved user sample from {SAMPLE_PATH}")

    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []
    for group, uids in saved_sample.items():
        user_data[group] = {}
        for uid in uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data on reload"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))

    print(f"  Loaded users per group:")
    for group in saved_sample:
        print(f"    {group}: {len(user_data[group])}/{len(saved_sample[group])}")
    print(f"  Total: {sum(len(user_data[g]) for g in user_data)}")
    if failed_users:
        print(f"  Failed on reload: {len(failed_users)}")

else:
    # --- FIRST RUN: load all users, sample, then save ---
    print(f"\nNo saved sample for {FINAL_TAG}, loading all users...")
    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []

    for group in user_splits:
        user_data[group] = {}
        group_uids = user_splits[group]
        for uid in group_uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))
                continue

    print(f"\nLoaded users per group:")
    total_valid = 0
    for group in user_splits:
        n = len(user_data[group])
        total_valid += n
        print(f"  {group}: {n}/{len(user_splits[group])} valid")
    print(f"  Total valid: {total_valid}")
    print(f"  Failed: {len(failed_users)}")

    # Proportional sampling if N_USERS is set
    if N_USERS is not None:
        group_sizes = {g: len(user_data[g]) for g in user_splits}
        total_available = sum(group_sizes.values())

        # Step 1: Keep proportions between train and test groups
        train_available = sum(group_sizes.get(g, 0) for g in TRAIN_GROUPS)
        test_available = sum(group_sizes.get(g, 0) for g in TEST_GROUPS)
        train_slots = int(round(N_USERS * train_available / total_available))
        test_slots = N_USERS - train_slots

        # Step 2: Train groups — prioritize u_train first, fill remainder with au_train
        n_u_train = min(group_sizes.get("u_train", 0), train_slots)
        n_au_train = min(group_sizes.get("au_train", 0), train_slots - n_u_train)
        raw_alloc = {"u_train": n_u_train, "au_train": n_au_train}

        # Step 3: Test groups — proportional allocation within test slots
        test_group_sizes = {g: group_sizes.get(g, 0) for g in TEST_GROUPS if group_sizes.get(g, 0) > 0}
        total_test_available = sum(test_group_sizes.values())
        for g in TEST_GROUPS:
            if total_test_available > 0 and group_sizes.get(g, 0) > 0:
                raw_alloc[g] = int(round(test_slots * group_sizes[g] / total_test_available))
            else:
                raw_alloc[g] = 0
        # Adjust test rounding to hit exact test_slots
        test_diff = test_slots - sum(raw_alloc.get(g, 0) for g in TEST_GROUPS)
        for g in sorted(TEST_GROUPS, key=lambda g: group_sizes.get(g, 0), reverse=True):
            if test_diff == 0:
                break
            adjustment = 1 if test_diff > 0 else -1
            raw_alloc[g] = max(1, raw_alloc[g] + adjustment)
            test_diff -= adjustment

        # Sample from each group
        sampled_user_data = {}
        for g in user_splits:
            if g not in raw_alloc or raw_alloc[g] == 0:
                sampled_user_data[g] = {}
                continue
            uids = list(user_data[g].keys())
            n_sample = min(raw_alloc[g], len(uids))
            sampled_uids = sample(uids, n_sample)
            sampled_user_data[g] = {uid: user_data[g][uid] for uid in sampled_uids}
        user_data = sampled_user_data

        print(f"\nSampled {N_USERS} users (train priority: u_train first, then au_train):")
        for g in user_splits:
            print(f"  {g}: {len(user_data[g])} (target {raw_alloc.get(g, 0)})")
        print(f"  Total sampled: {sum(len(user_data[g]) for g in user_splits)}")

    # Save the sample (user IDs per group) for reproducibility
    sample_to_save = {g: list(user_data[g].keys()) for g in user_data}
    with open(SAMPLE_PATH, "w") as f:
        json.dump(sample_to_save, f, indent=2)
    print(f"  Saved user sample to {SAMPLE_PATH}")

# Flat list of train-group users (for GNN training)
valid_users = list(user_data.get("u_train", {}).keys()) + list(user_data.get("au_train", {}).keys())
# Baseline SVC uses only u_train users
baseline_users = list(user_data.get("u_train", {}).keys())
print(f"\nTrain-group valid users: {len(valid_users)} (baseline SVC: {len(baseline_users)} from u_train only)")

## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [0]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

# ---------------------------------------------------------------------------
# User-level baseline cache (shared across experiments)
# Each user's result is stored independently so we never recompute a user.
# ---------------------------------------------------------------------------
BASELINE_USER_CACHE_PATH = f"{DATA_PATH}/baseline_svc_user_cache.pkl"

# Initialize cache from v3_full_N150 if it doesn't exist yet
if not os.path.exists(BASELINE_USER_CACHE_PATH):
    _init_path = "./experiments/v3_full_N150/baseline_svc_results.pkl"
    if os.path.exists(_init_path):
        print(f"Initializing user-level baseline cache from {_init_path}...")
        with open(_init_path, "rb") as f:
            _init_data = pickle.load(f)
        _cache = {}
        _init_f1s = _init_data["baseline_f1s"]
        _init_params = _init_data["baseline_best_params"]
        _init_preds = _init_data["all_baseline_test_preds"]
        # all_baseline_test_preds is ordered parallel to baseline_f1s keys
        _user_ids = list(_init_f1s.keys())
        for idx, uid in enumerate(_user_ids):
            preds, labels = _init_preds[idx]
            _cache[uid] = {
                "f1": _init_f1s[uid],
                "best_params": _init_params[uid],
                "preds": preds,
                "labels": labels,
            }
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump(_cache, f)
        print(f"  Initialized cache with {len(_cache)} users from v3_full_N150.")
        del _init_data, _cache, _init_f1s, _init_params, _init_preds
    else:
        print(f"No v3_full_N150 results found at {_init_path}, starting empty cache.")
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump({}, f)

# Load existing user-level cache
with open(BASELINE_USER_CACHE_PATH, "rb") as f:
    baseline_user_cache = pickle.load(f)
print(f"Baseline user cache: {len(baseline_user_cache)} users already computed.")

# Determine which baseline users still need processing
users_to_compute = [uid for uid in baseline_users if uid not in baseline_user_cache]
users_cached = [uid for uid in baseline_users if uid in baseline_user_cache]
print(f"  This experiment: {len(baseline_users)} baseline users")
print(f"  Already cached:  {len(users_cached)}")
print(f"  Need computing:  {len(users_to_compute)}")

if users_to_compute:
    # Reduced hyperparameter grid for faster iteration
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    t0 = time.time()
    for i, uid in enumerate(users_to_compute):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data["u_train"][uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        # Save to cache immediately
        baseline_user_cache[uid] = {
            "f1": best_f1,
            "best_params": best_params,
            "preds": best_preds,
            "labels": np.array(y_te),
        }

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(users_to_compute) - i - 1)
        print(f"  [{i+1:>3}/{len(users_to_compute)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {int(elapsed)//60}m{int(elapsed)%60:02d}s | ETA {int(remaining)//60}m{int(remaining)%60:02d}s)")

    total_time = time.time() - t0
    print(f"\nComputed {len(users_to_compute)} new users in {total_time:.1f}s "
          f"({total_time/len(users_to_compute):.1f}s/user avg).")

    # Persist updated cache
    with open(BASELINE_USER_CACHE_PATH, "wb") as f:
        pickle.dump(baseline_user_cache, f)
    print(f"  Cache updated: {len(baseline_user_cache)} total users.")
else:
    print("All baseline users already cached — no computation needed.")

# --- Assemble experiment-level results from cache ---
baseline_f1s = {uid: baseline_user_cache[uid]["f1"] for uid in baseline_users}
baseline_best_params = {uid: baseline_user_cache[uid]["best_params"] for uid in baseline_users}
all_baseline_test_preds = [
    (baseline_user_cache[uid]["preds"], baseline_user_cache[uid]["labels"])
    for uid in baseline_users
]

# Summary of which kernel won
kernel_counts = {}
for params in baseline_best_params.values():
    k = params[0]
    kernel_counts[k] = kernel_counts.get(k, 0) + 1
print(f"\nBaseline results for {len(baseline_users)} users — kernel distribution: {kernel_counts}")

In [0]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

GNN samples are created **on-the-fly** from user dataframes (no caching/storage of samples):
- **Training**: iterates users in shuffled chunks of `TRAIN_CHUNK_SIZE`, transforms to GNN samples,
  yields shuffled batches within each chunk, frees memory after each chunk.
- **Validation**: pre-computes samples (capped at `MAX_VAL_SAMPLES`, proportional across users),
  cached in CPU memory for fast repeated evaluation.

Architecture from 2.0, with aggressive anti-overfitting (test set has entirely unseen users):
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `dropout=0.5, drop_edge_rate=0.1` (stochastic regularization)
- `gate_param init=0.0` (sigmoid=0.5, balanced — shortcut generalizes better to unseen users)
- `weight_decay=1e-3` (L2)
- `lr=3e-3` with 5-epoch linear warmup + cosine decay
- `epochs=60, batch_size=256, log_every_n_steps=300`

In [0]:
from gnn_models import (
    PretrainedEmbeddingLookup, RetweetGNN,
    soft_f1_loss, combined_loss, evaluate, train_model,
)

In [0]:
# ---------------------------------------------------------------------------
# Build user assignments for training and validation.
# GNN samples are created on-the-fly by the loaders (no caching).
# ---------------------------------------------------------------------------
# BATCH_SIZE is set in the Parameters cell

# Train: all users from TRAIN_GROUPS (their "train" split)
train_user_items = []
for group in TRAIN_GROUPS:
    for uid in user_data.get(group, {}):
        train_user_items.append((group, uid))

# Val: train-group users' "test" split + test-group users' both splits
val_user_splits = []
for group in TRAIN_GROUPS:
    for uid in user_data.get(group, {}):
        val_user_splits.append((group, uid, "test"))
for group in TEST_GROUPS:
    for uid in user_data.get(group, {}):
        val_user_splits.append((group, uid, "train"))
        val_user_splits.append((group, uid, "test"))

print(f"Train users: {len(train_user_items)} (from {TRAIN_GROUPS})")
print(f"Val user/split combos: {len(val_user_splits)} "
      f"(from {TRAIN_GROUPS} test + {TEST_GROUPS} both)")
print(f"\nGNN samples will be created on-the-fly (chunk_size={TRAIN_CHUNK_SIZE}).")
print(f"Val samples capped at {MAX_VAL_SAMPLES} (proportional across users).")

In [0]:
import gc
from random import shuffle as _shuffle_list
from torch_geometric.data import Data, Batch


# ---------------------------------------------------------------------------
# Helper: convert a raw GNN sample dict to a PyG Data object (CPU tensors)
# ---------------------------------------------------------------------------
def _sample_to_pyg_data(sample):
    """Convert a raw sample dict from create_gnn_train_val_samples to PyG Data."""
    central_id = int(sample["central_user_id"])
    neighbor_ids = (
        sample["neighbor_ids"].tolist()
        if hasattr(sample["neighbor_ids"], "tolist")
        else list(sample["neighbor_ids"])
    )
    all_ids = [central_id] + neighbor_ids
    num_nodes = len(all_ids)

    user_ids = torch.tensor(all_ids, dtype=torch.long)

    retweeted_raw = sample["retweeted_ids"]
    retweeted_set = set(
        int(r) for r in (retweeted_raw.tolist() if hasattr(retweeted_raw, "tolist") else retweeted_raw)
    )
    retweet_flag = torch.tensor(
        [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
        dtype=torch.float,
    ).unsqueeze(1)

    ei = sample["edge_index"]
    if hasattr(ei, "__len__") and len(ei) > 0:
        ei_arr = np.array(ei, dtype=np.int64) if not isinstance(ei, np.ndarray) else ei.astype(np.int64)
        edge_index = torch.from_numpy(ei_arr).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    label = torch.tensor(int(sample["label"]), dtype=torch.long)

    return Data(
        user_ids=user_ids,
        retweet_flag=retweet_flag,
        edge_index=edge_index,
        y=label,
        num_nodes=num_nodes,
        central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(
            0, torch.tensor([0]), True
        ),
    )


# ---------------------------------------------------------------------------
# Shared helper: proportional allocation with largest-remainder method
# ---------------------------------------------------------------------------
def _proportional_allocate(row_counts, max_samples):
    """Proportionally allocate max_samples across items using largest-remainder.

    Each item gets at least 1 sample. Returns a list of allocations
    (same length as row_counts). If total available <= max_samples or
    max_samples is None, returns row_counts as-is.
    """
    total_available = sum(row_counts)
    if not max_samples or total_available <= max_samples:
        return list(row_counts)

    sample_rate = max_samples / total_available
    raw = [n * sample_rate for n in row_counts]
    allocations = [max(1, int(r)) for r in raw]

    # Fill deficit by giving +1 to items that lost the most from int() truncation
    deficit = max_samples - sum(allocations)
    if deficit > 0:
        remainders = sorted(
            range(len(raw)),
            key=lambda i: raw[i] - int(raw[i]),
            reverse=True,
        )
        for i in remainders[:deficit]:
            allocations[i] += 1

    # Trim any excess from max(1, ...) bumps on tiny items
    while sum(allocations) > max_samples:
        max_idx = max(range(len(allocations)), key=lambda i: allocations[i])
        allocations[max_idx] -= 1

    return allocations


# ---------------------------------------------------------------------------
# ChunkedGNNTrainLoader: on-the-fly transformation in user chunks
# ---------------------------------------------------------------------------
class ChunkedGNNTrainLoader:
    """Training loader that transforms GNN samples on-the-fly in user chunks.

    Each epoch:
      1. Shuffles the user list
      2. Processes users in chunks of `chunk_size`
      3. For each chunk: transforms all users' training data to GNN samples,
         shuffles the combined samples, yields batches of `batch_size`
      4. Frees chunk samples after processing

    All tensors are kept on CPU. The training loop moves batches to GPU.
    """

    def __init__(self, user_items, user_data, graph, batch_size, chunk_size=10, target_pos_rate=None):
        self.user_items = list(user_items)
        self.user_data = user_data
        self.graph = graph
        self.batch_size = batch_size
        self.chunk_size = chunk_size
        self.target_pos_rate = target_pos_rate
        self._failed_users = []
        self._estimated_samples = None

    @property
    def total_samples(self):
        if self._estimated_samples is None:
            self._estimated_samples = sum(self.get_label_counts().values())
        return self._estimated_samples

    def __len__(self):
        return self.total_samples // self.batch_size

    def __iter__(self):
        """Yield PyG Batch objects, processing users in chunks."""
        users = list(self.user_items)
        _shuffle_list(users)

        total_yielded = 0
        self._failed_users = []

        for chunk_start in range(0, len(users), self.chunk_size):
            chunk_users = users[chunk_start : chunk_start + self.chunk_size]

            # Transform all users in this chunk to GNN samples
            chunk_data_list = []
            for group, uid in chunk_users:
                X_tr, X_te, y_tr, y_te = self.user_data[group][uid]
                try:
                    train_samples, _ = create_gnn_train_val_samples(
                        uid, self.graph, X_tr, y_tr, X_te, y_te
                    )
                    for s in train_samples:
                        chunk_data_list.append(_sample_to_pyg_data(s))
                except Exception as e:
                    self._failed_users.append((group, uid, str(e)))
                    continue

            if not chunk_data_list:
                continue

            # --- Oversample positives to reach target_pos_rate ---
            if self.target_pos_rate is not None:
                pos_idx = [i for i, d in enumerate(chunk_data_list) if d.y.item() == 1]
                n_pos = len(pos_idx)
                n_neg = len(chunk_data_list) - n_pos
                if n_pos > 0 and n_neg > 0:
                    target_n_pos = int(np.ceil(
                        self.target_pos_rate * n_neg / (1.0 - self.target_pos_rate)
                    ))
                    n_extra = target_n_pos - n_pos
                    if n_extra > 0:
                        extra_idx = np.random.choice(pos_idx, size=n_extra, replace=True)
                        extra_samples = [chunk_data_list[i] for i in extra_idx]
                        chunk_data_list.extend(extra_samples)

            # Shuffle within chunk
            _shuffle_list(chunk_data_list)

            # Yield batches
            for start in range(0, len(chunk_data_list) - self.batch_size + 1, self.batch_size):
                batch = Batch.from_data_list(chunk_data_list[start : start + self.batch_size])
                yield batch
                total_yielded += self.batch_size

            # Free chunk memory
            del chunk_data_list
            gc.collect()

        # Update estimate for __len__ after first full epoch
        if total_yielded > 0:
            self._estimated_samples = total_yielded

    def get_label_counts(self):
        """Count labels from y_tr, adjusted for oversampling if target_pos_rate is set."""
        counts = {}
        for group, uid in self.user_items:
            _, _, y_tr, _ = self.user_data[group][uid]
            y_arr = np.asarray(y_tr).ravel()
            for label in y_arr:
                counts[int(label)] = counts.get(int(label), 0) + 1
        # Adjust for oversampling
        if self.target_pos_rate is not None and 1 in counts and 0 in counts:
            n_neg = counts[0]
            target_n_pos = int(np.ceil(
                self.target_pos_rate * n_neg / (1.0 - self.target_pos_rate)
            ))
            if target_n_pos > counts[1]:
                counts[1] = target_n_pos
        return counts


# ---------------------------------------------------------------------------
# CachedValLoader: streams val samples from disk one chunk at a time
# ---------------------------------------------------------------------------
class CachedValLoader:
    """Validation loader that keeps only ONE chunk in memory at a time.

    During creation: saves each chunk to disk as soon as it's generated,
    freeing memory before computing the next chunk.
    During iteration: loads one chunk file at a time, yields batches from it,
    then frees it before loading the next.

    If max_samples is set, subsamples input rows BEFORE GNN transformation
    (proportionally across users) to avoid wasting compute on samples that
    would be discarded.
    """

    CHUNK_SIZE = 1000  # samples per chunk file

    def __init__(self, val_user_splits, user_data, graph, batch_size, max_samples=None, cache_path=None):
        self.batch_size = batch_size
        self._cache_path = cache_path
        self._total_samples = 0
        self._label_counts = {}
        self._failed_users = []
        self._chunk_files = []  # sorted list of chunk file paths

        # Try loading from existing chunked cache directory
        if cache_path and os.path.isdir(cache_path):
            meta_path = os.path.join(cache_path, "metadata.json")
            chunk_files = sorted(
                f for f in os.listdir(cache_path)
                if f.startswith("chunk_") and f.endswith(".pt")
            )
            if chunk_files and os.path.exists(meta_path):
                import json
                with open(meta_path, "r") as f:
                    meta = json.load(f)
                self._chunk_files = [os.path.join(cache_path, cf) for cf in chunk_files]
                self._total_samples = meta["total_samples"]
                self._label_counts = {int(k): v for k, v in meta["label_counts"].items()}
                print(f"Loaded val cache metadata from: {cache_path}")
                print(f"  {self._total_samples} samples, {len(self._chunk_files)} chunks, {len(self)} batches")
                return

        # --- Build cache from scratch, saving chunks incrementally ---
        assert cache_path, "cache_path is required for CachedValLoader (streaming mode)"
        os.makedirs(cache_path, exist_ok=True)

        n_total = len(val_user_splits)
        print(f"Pre-computing validation samples ({n_total} user/split combos)...")

        # Count available rows per user/split (cheap — just .shape[0])
        row_counts = []
        for group, uid, split_name in val_user_splits:
            X_tr, X_te, _, _ = user_data[group][uid]
            row_counts.append(X_tr.shape[0] if split_name == "train" else X_te.shape[0])

        total_available_rows = sum(row_counts)

        # Determine per-user/split allocation (largest-remainder method)
        allocations = _proportional_allocate(row_counts, max_samples)
        do_subsample = bool(max_samples) and total_available_rows > max_samples
        if do_subsample:
            print(f"  Sampling enabled: {max_samples} target from {total_available_rows} available rows"
                  f" (allocated {sum(allocations)})")

        # Transform users and flush chunks to disk incrementally
        pending_samples = []  # accumulate up to CHUNK_SIZE before flushing
        chunk_idx = 0

        def _flush_chunk():
            nonlocal pending_samples, chunk_idx
            if not pending_samples:
                return
            chunk_path = os.path.join(cache_path, f"chunk_{chunk_idx:03d}.pt")
            torch.save(pending_samples, chunk_path)
            self._chunk_files.append(chunk_path)
            self._total_samples += len(pending_samples)
            for data in pending_samples:
                label = data.y.item()
                self._label_counts[label] = self._label_counts.get(label, 0) + 1
            chunk_idx += 1
            pending_samples = []
            gc.collect()

        for i, (group, uid, split_name) in enumerate(val_user_splits):
            if (i + 1) % 5 == 0 or (i + 1) == n_total:
                print(
                    f"  [{i+1}/{n_total}] {self._total_samples + len(pending_samples)} samples, "
                    f"{chunk_idx} chunks saved...",
                    end="\r",
                )

            X_tr, X_te, y_tr, y_te = user_data[group][uid]
            n_keep = allocations[i]

            try:
                if do_subsample:
                    if split_name == "test":
                        idx = np.random.choice(X_te.shape[0], size=min(n_keep, X_te.shape[0]), replace=False)
                        _, test_samples = create_gnn_train_val_samples(
                            uid, graph, X_tr.iloc[:1], y_tr.iloc[:1], X_te.iloc[idx], y_te.iloc[idx]
                        )
                        samples = test_samples
                    else:
                        idx = np.random.choice(X_tr.shape[0], size=min(n_keep, X_tr.shape[0]), replace=False)
                        train_samples, _ = create_gnn_train_val_samples(
                            uid, graph, X_tr.iloc[idx], y_tr.iloc[idx], X_te.iloc[:1], y_te.iloc[:1]
                        )
                        samples = train_samples
                else:
                    train_samples, test_samples = create_gnn_train_val_samples(
                        uid, graph, X_tr, y_tr, X_te, y_te
                    )
                    samples = train_samples if split_name == "train" else test_samples

                for s in samples:
                    pending_samples.append(_sample_to_pyg_data(s))
                    # Flush when chunk is full
                    if len(pending_samples) >= self.CHUNK_SIZE:
                        _flush_chunk()
            except Exception as e:
                self._failed_users.append((group, uid, str(e)))
                continue

        # Flush any remaining samples
        _flush_chunk()
        print()  # newline after \r progress

        # Save metadata so future loads are instant (no chunk deserialization)
        import json
        meta_path = os.path.join(cache_path, "metadata.json")
        with open(meta_path, "w") as f:
            json.dump({"total_samples": self._total_samples, "label_counts": self._label_counts}, f)

        gc.collect()
        print(
            f"  Saved {self._total_samples} val samples in {len(self._chunk_files)} chunks to: {cache_path}"
            f" (from {n_total - len(self._failed_users)} user/split combos, "
            f"rows available: {total_available_rows}, failed: {len(self._failed_users)})"
        )

    @property
    def total_samples(self):
        return self._total_samples

    def __len__(self):
        return -(-self._total_samples // self.batch_size)  # ceiling div (includes tail batch)

    def __iter__(self):
        """Stream batches loading one chunk at a time."""
        carry_over = []  # leftover samples from previous chunk (< batch_size)
        for chunk_path in self._chunk_files:
            chunk_data = torch.load(chunk_path, map_location="cpu", weights_only=False)
            samples = carry_over + chunk_data
            del chunk_data

            # Yield full batches from this chunk
            start = 0
            while start + self.batch_size <= len(samples):
                batch = Batch.from_data_list(samples[start : start + self.batch_size])
                yield batch
                start += self.batch_size

            # Keep remainder for next chunk
            carry_over = samples[start:]
            del samples
            gc.collect()

        # Yield remaining tail as a final (possibly smaller) batch
        if carry_over:
            yield Batch.from_data_list(carry_over)

    def get_label_counts(self):
        return dict(self._label_counts)


In [0]:
train_loader = ChunkedGNNTrainLoader(
    train_user_items, user_data, graph,
    batch_size=BATCH_SIZE, chunk_size=TRAIN_CHUNK_SIZE,
    target_pos_rate=TARGET_POS_RATE,
)

print(f"Train loader: {len(train_user_items)} users, "
      f"chunk_size={TRAIN_CHUNK_SIZE}, batch_size={BATCH_SIZE}")

# Compute class weights from train labels
print("\nComputing class weights from train labels...")
label_counts = train_loader.get_label_counts()
total_train = sum(label_counts.values())
classes = np.array(sorted(label_counts.keys()))
class_weights = torch.tensor(
    [total_train / (len(classes) * label_counts[c]) for c in classes],
    dtype=torch.float,
)
_train_pos_rate = label_counts.get(1, 0) / max(total_train, 1)
print(f"  Label counts: {label_counts}")
print(f"  Positive rate: {_train_pos_rate:.4f} ({label_counts.get(1, 0)}/{total_train})")
print(f"  Class weights: {class_weights.tolist()}")
print(f"  Total train samples (from y_tr): {total_train}")

In [0]:
VAL_CACHE_PATH = os.path.join(EXPERIMENT_DIR, f"val_samples_cache_{MAX_VAL_SAMPLES}") if MAX_VAL_SAMPLES else None

val_loader = CachedValLoader(
    val_user_splits, user_data, graph,
    batch_size=BATCH_SIZE, max_samples=MAX_VAL_SAMPLES,
    cache_path=VAL_CACHE_PATH,
)
_val_lc = val_loader.get_label_counts()
_val_pos_rate = _val_lc.get(1, 0) / max(sum(_val_lc.values()), 1)
print(f"\nVal loader: {val_loader.total_samples} cached samples, {len(val_loader)} batches")
print(f"  Positive rate: {_val_pos_rate:.4f} ({_val_lc.get(1, 0)}/{sum(_val_lc.values())})")

In [0]:
# ---------------------------------------------------------------------------
# SampledTrainEvalLoader: lightweight on-the-fly loader for train-set evaluation.
# Only saves sampling metadata (user + row indices) to disk — no GNN samples.
# ---------------------------------------------------------------------------
class SampledTrainEvalLoader:
    """Train-eval loader that samples a fixed subset of rows per user.

    On first creation: proportionally allocates rows across users up to
    max_samples, randomly selects row indices, and saves only the metadata
    (JSON) for reproducibility. No GNN samples are stored on disk.

    During iteration: generates GNN samples on-the-fly from the saved
    row indices, processes users in chunks to limit memory usage.
    """

    CHUNK_SIZE = 10  # users per chunk during iteration

    def __init__(self, train_user_items, user_data, graph, batch_size,
                 max_samples, metadata_path):
        self.user_data = user_data
        self.graph = graph
        self.batch_size = batch_size
        self._failed_users = []
        self._total_samples = 0
        self._label_counts = {}

        # Load or create sampling metadata
        if os.path.exists(metadata_path):
            with open(metadata_path, "r") as f:
                meta = json.load(f)
            self._allocations = meta["allocations"]  # [{group, uid, row_indices}, ...]
            self._total_samples = meta["total_samples"]
            self._label_counts = {int(k): v for k, v in meta["label_counts"].items()}
            print(f"Loaded train-eval metadata from: {metadata_path}")
            print(f"  {self._total_samples} samples across {len(self._allocations)} users")
        else:
            # Determine per-user row counts
            row_counts = []
            for group, uid in train_user_items:
                X_tr, _, _, _ = user_data[group][uid]
                row_counts.append(X_tr.shape[0])

            total_available = sum(row_counts)

            # Proportional allocation (largest-remainder method, shared helper)
            allocations = _proportional_allocate(row_counts, max_samples)

            # Sample row indices per user and compute label counts
            self._allocations = []
            for i, (group, uid) in enumerate(train_user_items):
                n_keep = allocations[i]
                X_tr, _, y_tr, _ = user_data[group][uid]
                n_available = X_tr.shape[0]
                idx = np.random.choice(n_available, size=min(n_keep, n_available), replace=False)
                idx_sorted = sorted(idx.tolist())
                self._allocations.append({"group": group, "uid": uid, "row_indices": idx_sorted})

                # Count labels from the sampled rows
                y_arr = np.asarray(y_tr).ravel()
                for label in y_arr[idx]:
                    self._label_counts[int(label)] = self._label_counts.get(int(label), 0) + 1

            self._total_samples = sum(len(a["row_indices"]) for a in self._allocations)

            # Save metadata
            meta = {
                "total_samples": self._total_samples,
                "label_counts": self._label_counts,
                "allocations": self._allocations,
            }
            with open(metadata_path, "w") as f:
                json.dump(meta, f)
            print(f"Created train-eval metadata: {metadata_path}")
            print(f"  {self._total_samples} samples from {total_available} available "
                  f"({len(self._allocations)} users)")

    @property
    def total_samples(self):
        return self._total_samples

    def __len__(self):
        return -(-self._total_samples // self.batch_size)  # ceiling div (includes tail batch)

    def get_label_counts(self):
        return dict(self._label_counts)

    def __iter__(self):
        """Yield PyG Batch objects, generating GNN samples on-the-fly in chunks.

        Uses a cross-chunk buffer to avoid losing tail samples per chunk.
        """
        self._failed_users = []
        buffer = []

        for chunk_start in range(0, len(self._allocations), self.CHUNK_SIZE):
            chunk_allocs = self._allocations[chunk_start:chunk_start + self.CHUNK_SIZE]

            for alloc in chunk_allocs:
                group, uid = alloc["group"], alloc["uid"]
                row_indices = alloc["row_indices"]
                X_tr, X_te, y_tr, y_te = self.user_data[group][uid]

                try:
                    # Subsample train rows using saved indices
                    X_sub = X_tr.iloc[row_indices]
                    y_sub = y_tr.iloc[row_indices]
                    train_samples, _ = create_gnn_train_val_samples(
                        uid, self.graph, X_sub, y_sub, X_te.iloc[:1], y_te.iloc[:1]
                    )
                    for s in train_samples:
                        buffer.append(_sample_to_pyg_data(s))
                except Exception as e:
                    self._failed_users.append((group, uid, str(e)))
                    continue

            # Flush full batches from buffer to keep memory bounded
            while len(buffer) >= self.batch_size:
                batch = Batch.from_data_list(buffer[:self.batch_size])
                buffer = buffer[self.batch_size:]
                yield batch

        # Yield remaining tail as a final (possibly smaller) batch
        if buffer:
            yield Batch.from_data_list(buffer)


# --- Instantiate ---
TRAIN_EVAL_META_PATH = os.path.join(EXPERIMENT_DIR, f"train_eval_metadata_{MAX_TRAIN_EVAL_SAMPLES}.json")

train_eval_loader = SampledTrainEvalLoader(
    train_user_items, user_data, graph,
    batch_size=BATCH_SIZE, max_samples=MAX_TRAIN_EVAL_SAMPLES,
    metadata_path=TRAIN_EVAL_META_PATH,
)
_tr_eval_lc = train_eval_loader.get_label_counts()
_tr_eval_pos_rate = _tr_eval_lc.get(1, 0) / max(sum(_tr_eval_lc.values()), 1)
print(f"\nTrain-eval loader: {train_eval_loader.total_samples} samples, {len(train_eval_loader)} batches")
print(f"  Positive rate: {_tr_eval_pos_rate:.4f} ({_tr_eval_lc.get(1, 0)}/{sum(_tr_eval_lc.values())})")

In [0]:
# Anti-overfitting: strong regularization to prevent memorizing train users' graph patterns.
# Test set has entirely unseen users — model must generalize graph structure, not memorize it.
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=DROPOUT,       # from tuned hparams or default
    drop_edge_rate=DROP_EDGE_RATE,  # from tuned hparams or default
).to(device)

# Only apply INIT_WEIGHTS_PATH when starting fresh (no existing checkpoint from this experiment).
# This prevents accidentally re-initializing from pre-trained weights when resuming a run.
_existing_checkpoint = os.path.join(EXPERIMENT_DIR, "best_retweet_gnn_general.pt")
_is_fresh_start = RESET_GNN_TRAINING or not os.path.exists(_existing_checkpoint)
_apply_init_weights = bool(INIT_WEIGHTS_PATH) and _is_fresh_start

if not _is_fresh_start and INIT_WEIGHTS_PATH:
    print(f"⚠️  Existing checkpoint found at {_existing_checkpoint} — skipping INIT_WEIGHTS_PATH.")
    print(f"   (Set RESET_GNN_TRAINING=True to force re-initialization from pre-trained weights.)")

if _apply_init_weights:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
    state_dict = torch.load(INIT_WEIGHTS_PATH, map_location=device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Missing keys (will use random init): {missing}")
    if unexpected:
        print(f"  Unexpected keys (ignored): {unexpected}")
    print(f"  Weights loaded successfully. gate_param = {model.gate_param.item():.4f}")
else:
    # Gate at 0.0 → sigmoid=0.5 (balanced start). Let the model earn GNN contribution.
    # Shortcut head uses aggregate stats (rt_frac, node_count) which generalize better to unseen users.
    # If GNN can't beat shortcut on val, gate will stay low — that's fine.
    with torch.no_grad():
        model.gate_param.fill_(0.0)

In [0]:
# user_data and graph are kept alive — needed by ChunkedGNNTrainLoader each epoch
gc.collect()
torch.cuda.empty_cache()

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

# Set MLflow experiment (grouped by project, one run per training execution)
# Use explicit get-or-create to avoid a bug in mlflow.set_experiment() where
# create_experiment errors (other than RESOURCE_ALREADY_EXISTS) are silently
# swallowed, leaving experiment_id=None and producing a misleading
# "For input string: None" error.
EXPERIMENT_NAME = f"/Users/pablo.celayes@bolt.eu/sna_classifier_gnn/{FINAL_TAG}"
os.makedirs(f"/Workspace{os.path.dirname(EXPERIMENT_NAME)}", exist_ok=True)
_client = MlflowClient()
_exp = _client.get_experiment_by_name(EXPERIMENT_NAME)
if _exp is None:
    _exp_id = _client.create_experiment(EXPERIMENT_NAME)
else:
    _exp_id = _exp.experiment_id
mlflow.set_experiment(experiment_id=_exp_id)

# Start run
mlflow_run = mlflow.start_run(run_name=FINAL_TAG)

# --- Log training hyperparameters ---
# On resume, read from saved config (same source cell 25 will use) so MLflow
# logs the actual fine-tuning values, not the base defaults.
_base_lr = 3e-3
_base_wd = 1e-3
_training_config_path = os.path.join(EXPERIMENT_DIR, "training_config.json")
_checkpoint_exists = os.path.exists(os.path.join(EXPERIMENT_DIR, "training_checkpoint.pt"))
if not RESET_GNN_TRAINING and _checkpoint_exists and os.path.exists(_training_config_path):
    with open(_training_config_path) as _f:
        _saved_cfg = json.load(_f)
    _effective_lr = _saved_cfg["lr"]
    _effective_wd = _saved_cfg["weight_decay"]
    _effective_warmup = _saved_cfg["warmup_epochs"]
else:
    if _TUNED_LR is not None:
        _effective_lr = _TUNED_LR
        _effective_wd = _TUNED_WD
        _effective_warmup = FINETUNE_WARMUP_EPOCHS
    else:
        _effective_lr = _base_lr * FINETUNE_LR_FACTOR if _apply_init_weights else _base_lr
        _effective_wd = _base_wd * FINETUNE_WD_FACTOR if _apply_init_weights else _base_wd
        _effective_warmup = FINETUNE_WARMUP_EPOCHS if _apply_init_weights else 5

mlflow.log_params({
    # Experiment
    "experiment_tag": EXPERIMENT_TAG,
    "final_tag": FINAL_TAG,
    "n_users": N_USERS if N_USERS else "all",
    "n_train_users": len(valid_users),
    # Training schedule
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "train_chunk_size": TRAIN_CHUNK_SIZE,
    "lr": _effective_lr,
    "weight_decay": _effective_wd,
    "warmup_epochs": _effective_warmup,
    "patience": PATIENCE if PATIENCE else "disabled",
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS or 1,
    "mixed_precision": MIXED_PRECISION,
    # Eval
    "log_every_n_steps": LOG_EVERY_N_STEPS,
    "train_f1_every_n_epochs": TRAIN_F1_EVERY_N_EPOCHS or "disabled",
    "max_val_samples": MAX_VAL_SAMPLES or "all",
    "max_train_eval_samples": MAX_TRAIN_EVAL_SAMPLES or "all",
    # Model architecture
    "ff_hidden_dim": 64,
    "gcn_hidden_dim": 64,
    "transformer_dim": 64,
    "transformer_heads": 4,
    "dropout": DROPOUT,
    "drop_edge_rate": DROP_EDGE_RATE,
    # Init
    "init_weights_path": INIT_WEIGHTS_PATH or "none",
    "reset_gnn_training": RESET_GNN_TRAINING,
})

print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"MLflow run ID:     {mlflow_run.info.run_id}")

# ---------------------------------------------------------------------------
# Background logger: polls training_history.pkl and streams metrics to MLflow
# ---------------------------------------------------------------------------
import threading
import pickle as _pkl

class MLflowHistoryLogger(threading.Thread):
    """Daemon thread that polls training_history.pkl and logs new metrics to MLflow."""

    def __init__(self, history_path, poll_interval=30):
        super().__init__(daemon=True)
        self.history_path = history_path
        self.poll_interval = poll_interval
        self._stop_event = threading.Event()
        self._logged_checkpoint_steps = set()
        self._logged_epoch_steps = set()

    def stop(self):
        self._stop_event.set()

    def sync(self):
        """Read history file and log any new metrics to MLflow."""
        if not os.path.exists(self.history_path):
            return
        try:
            with open(self.history_path, "rb") as f:
                h = _pkl.load(f)
        except Exception:
            return  # file may be mid-write

        # Checkpoint-level metrics
        for i, step in enumerate(h.get("step", [])):
            if step not in self._logged_checkpoint_steps:
                mlflow.log_metrics({
                    "checkpoint/train_loss": h["train_loss"][i],
                    "checkpoint/val_loss": h["val_loss"][i],
                    "checkpoint/val_f1": h["val_f1"][i],
                }, step=step)
                self._logged_checkpoint_steps.add(step)

        # Epoch-level metrics
        for i, step in enumerate(h.get("epoch_step", [])):
            if step not in self._logged_epoch_steps:
                metrics = {
                    "epoch/val_loss": h["epoch_val_loss"][i],
                    "epoch/val_f1": h["epoch_val_f1"][i],
                }
                if i < len(h.get("epoch_train_f1", [])):
                    metrics["epoch/train_f1"] = h["epoch_train_f1"][i]
                mlflow.log_metrics(metrics, step=step)
                self._logged_epoch_steps.add(step)

    def run(self):
        while not self._stop_event.is_set():
            self.sync()
            self._stop_event.wait(self.poll_interval)
        self.sync()  # final sync on stop

In [0]:
# Train with on-the-fly GNN DataLoaders — anti-overfitting configuration:
#   - lr=3e-3 with 5-epoch linear warmup: prevents fast memorization in early steps
#   - weight_decay=1e-3: L2 prevents weight specialization
#   - dropout=0.5 + drop_edge=0.1: stochastic regularization
#   - gate_param=0.0 (50/50): shortcut generalizes better; GNN must earn its contribution
#   - ChunkedGNNTrainLoader shuffles user order + within-chunk each epoch

import sys
import time
from datetime import datetime
from contextlib import contextmanager

class TeeLogger:
    """Tee stdout to both the original stream and a timestamped log file."""
    def __init__(self, log_path, original_stdout):
        self._original = original_stdout
        self._file = open(log_path, "a", buffering=1)  # line-buffered
        self._line_buffer = ""

    def write(self, msg):
        self._original.write(msg)
        # Add timestamp at the start of each complete line
        for char in msg:
            if char == "\n":
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self._file.write(f"[{timestamp}] {self._line_buffer}\n")
                self._line_buffer = ""
            else:
                self._line_buffer += char

    def flush(self):
        self._original.flush()
        self._file.flush()

    def close(self):
        # Flush any remaining buffer
        if self._line_buffer:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            self._file.write(f"[{timestamp}] {self._line_buffer}\n")
            self._line_buffer = ""
        self._file.close()

@contextmanager
def tee_to_log(log_path):
    """Context manager that tees all stdout to a timestamped log file."""
    original_stdout = sys.stdout
    tee = TeeLogger(log_path, original_stdout)
    sys.stdout = tee
    try:
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()
        print(f"Training log saved to: {log_path}")

TRAINING_LOG_PATH = f"{EXPERIMENT_DIR}/training.log"

# Resolve effective hyperparameters: load from saved config on resume,
# compute fresh (with finetune factors if applicable) on first run.
_base_lr = 3e-3
_base_wd = 1e-3
_training_config_path = os.path.join(EXPERIMENT_DIR, "training_config.json")
_checkpoint_exists = os.path.exists(os.path.join(EXPERIMENT_DIR, "training_checkpoint.pt"))
_can_resume = not RESET_GNN_TRAINING and _checkpoint_exists

if _can_resume and os.path.exists(_training_config_path):
    # Resume: reload the exact hyperparameters used by the interrupted run
    with open(_training_config_path) as _f:
        _saved_config = json.load(_f)
    _train_lr = _saved_config["lr"]
    _train_wd = _saved_config["weight_decay"]
    _warmup_epochs = _saved_config["warmup_epochs"]
    print(f"Resuming with saved training config from {_training_config_path}:")
    print(f"  lr={_train_lr:.1e}, weight_decay={_train_wd:.1e}, warmup_epochs={_warmup_epochs}")
    if _saved_config.get("is_finetuning"):
        print(f"  (originally fine-tuned from {_saved_config.get('init_weights_path', '?')})")
else:
    if _TUNED_LR is not None:
        _train_lr = _TUNED_LR
        _train_wd = _TUNED_WD
        _warmup_epochs = FINETUNE_WARMUP_EPOCHS
        print(f"Using tuned hparams: lr={_train_lr:.2e}, wd={_train_wd:.2e}, warmup={_warmup_epochs}")
    else:
        _train_lr = _base_lr * FINETUNE_LR_FACTOR if _apply_init_weights else _base_lr
        _train_wd = _base_wd * FINETUNE_WD_FACTOR if _apply_init_weights else _base_wd
        _warmup_epochs = FINETUNE_WARMUP_EPOCHS if _apply_init_weights else 5
    # Persist config so interrupted runs resume with the same hyperparameters
    _config_to_save = {
        "lr": _train_lr,
        "weight_decay": _train_wd,
        "warmup_epochs": _warmup_epochs,
        "is_finetuning": bool(_apply_init_weights),
        "init_weights_path": INIT_WEIGHTS_PATH,
    }
    with open(_training_config_path, "w") as _f:
        json.dump(_config_to_save, _f, indent=2)
    print(f"Training config saved to {_training_config_path}")
    if _apply_init_weights:
        print(f"Fine-tuning mode:")
        print(f"  LR reduced from {_base_lr:.1e} to {_train_lr:.1e} (factor={FINETUNE_LR_FACTOR})")
        print(f"  Weight decay reduced from {_base_wd:.1e} to {_train_wd:.1e} (factor={FINETUNE_WD_FACTOR})")
        print(f"  Warmup epochs reduced from 5 to {_warmup_epochs}")

# Start MLflow background logger (polls training_history.pkl every 30s)
_history_pkl = os.path.join(EXPERIMENT_DIR, "training_history.pkl")
_mlflow_logger = MLflowHistoryLogger(_history_pkl, poll_interval=30)
_mlflow_logger.start()

# Ensure MLflow logger syncs even on crash/cancellation
import atexit
def _cleanup_mlflow_logger():
    if _mlflow_logger.is_alive():
        _mlflow_logger.stop()
        _mlflow_logger.join(timeout=5)
    _mlflow_logger.sync()
atexit.register(_cleanup_mlflow_logger)

with tee_to_log(TRAINING_LOG_PATH):
    # --- Initial evaluation before training (when using pretrained weights) ---
    if _apply_init_weights:
        from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
        print("\n" + "=" * 60)
        print("INITIAL EVALUATION (pre-trained weights, before training)")
        print("=" * 60)
        model.eval()
        _init_criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

        # Evaluate on validation set
        _val_preds, _val_labels, _val_loss, _val_n = [], [], 0.0, 0
        _t_val_init = time.time()
        with torch.no_grad():
            for _batch in val_loader:
                _batch = _batch.to(device)
                _out = model(_batch)
                _val_loss += _init_criterion(_out, _batch.y).item()
                _val_n += 1
                _val_preds.extend(_out.argmax(dim=1).cpu().tolist())
                _val_labels.extend(_batch.y.cpu().tolist())
        _val_init_elapsed = time.time() - _t_val_init
        print(f"  Val   — Loss: {_val_loss / max(_val_n, 1):.4f} | "
              f"Acc: {accuracy_score(_val_labels, _val_preds):.4f} | "
              f"F1: {f1_score(_val_labels, _val_preds):.4f} | "
              f"P: {precision_score(_val_labels, _val_preds, zero_division=0):.4f} | "
              f"R: {recall_score(_val_labels, _val_preds, zero_division=0):.4f} | "
              f"samples: {len(_val_labels)} | "
              f"took {int(_val_init_elapsed)//60}m {int(_val_init_elapsed)%60:02d}s")

        # Evaluate on training set (using sampled train-eval loader)
        _tr_preds, _tr_labels, _tr_loss, _tr_n = [], [], 0.0, 0
        _t_tr_init = time.time()
        with torch.no_grad():
            for _batch in train_eval_loader:
                _batch = _batch.to(device)
                _out = model(_batch)
                _tr_loss += _init_criterion(_out, _batch.y).item()
                _tr_n += 1
                _tr_preds.extend(_out.argmax(dim=1).cpu().tolist())
                _tr_labels.extend(_batch.y.cpu().tolist())
        _tr_init_elapsed = time.time() - _t_tr_init
        print(f"  Train — Loss: {_tr_loss / max(_tr_n, 1):.4f} | "
              f"Acc: {accuracy_score(_tr_labels, _tr_preds):.4f} | "
              f"F1: {f1_score(_tr_labels, _tr_preds):.4f} | "
              f"P: {precision_score(_tr_labels, _tr_preds, zero_division=0):.4f} | "
              f"R: {recall_score(_tr_labels, _tr_preds, zero_division=0):.4f} | "
              f"samples: {len(_tr_labels)} | "
              f"took {int(_tr_init_elapsed)//60}m {int(_tr_init_elapsed)%60:02d}s")
        print("=" * 60 + "\n")
        del _val_preds, _val_labels, _tr_preds, _tr_labels, _init_criterion
        gc.collect()
        torch.cuda.empty_cache()

    # Direct MLflow callback — logs metrics at each checkpoint without polling
    def _mlflow_on_checkpoint(history, global_step, is_epoch_end):
        try:
            metrics = {}
            if not is_epoch_end and history["step"]:
                i = len(history["step"]) - 1
                metrics["checkpoint/train_loss"] = history["train_loss"][i]
                metrics["checkpoint/val_loss"] = history["val_loss"][i]
                metrics["checkpoint/val_f1"] = history["val_f1"][i]
            if is_epoch_end and history["epoch_step"]:
                i = len(history["epoch_step"]) - 1
                metrics["epoch/val_loss"] = history["epoch_val_loss"][i]
                metrics["epoch/val_f1"] = history["epoch_val_f1"][i]
                if i < len(history.get("epoch_train_f1", [])) and history["epoch_train_f1"][i] is not None:
                    metrics["epoch/train_f1"] = history["epoch_train_f1"][i]
            if metrics:
                mlflow.log_metrics(metrics, step=global_step)
        except Exception as e:
            print(f"Warning: MLflow callback error: {e}")

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        experiment_dir=EXPERIMENT_DIR,
        class_weights=class_weights,
        epochs=EPOCHS,
        device=device,
        lr=_train_lr,
        log_every_n_steps=LOG_EVERY_N_STEPS,
        patience=PATIENCE,
        lr_warmup_epochs=_warmup_epochs,
        weight_decay=_train_wd,
        resume=_can_resume,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=MIXED_PRECISION,
        train_f1_every_n_epochs=TRAIN_F1_EVERY_N_EPOCHS,
        train_eval_loader=train_eval_loader,
        on_checkpoint=_mlflow_on_checkpoint,
    )

# Stop MLflow background logger and do final sync
_mlflow_logger.stop()
_mlflow_logger.join(timeout=10)
print(f"MLflow logger: logged {len(_mlflow_logger._logged_checkpoint_steps)} checkpoints, "
      f"{len(_mlflow_logger._logged_epoch_steps)} epochs during training.")

In [0]:
# ---------------------------------------------------------------------------
# Final sync: catch any metrics the background logger missed at the boundary
# ---------------------------------------------------------------------------
if '_mlflow_logger' in dir():
    _mlflow_logger.sync()

# Summary best metrics (non-step-indexed, appear as top-level run metrics)
if history:
    summary = {}
    if history.get("val_f1"):
        summary["best_checkpoint_val_f1"] = max(history["val_f1"])
    if history.get("epoch_val_f1"):
        summary["best_epoch_val_f1"] = max(history["epoch_val_f1"])
    if history.get("epoch_train_f1"):
        summary["best_epoch_train_f1"] = max(history["epoch_train_f1"])
    if history.get("train_loss"):
        summary["final_train_loss"] = history["train_loss"][-1]
    if history.get("val_loss"):
        summary["final_val_loss"] = history["val_loss"][-1]
    if summary:
        mlflow.log_metrics(summary)
        for k, v in summary.items():
            print(f"  {k}: {v:.4f}")

# Log artifacts: best model weights + training history
best_model_path = os.path.join(EXPERIMENT_DIR, "best_retweet_gnn_general.pt")
if os.path.exists(best_model_path):
    mlflow.log_artifact(best_model_path, artifact_path="model")
    print(f"Logged model artifact: {best_model_path}")

history_pkl_path = os.path.join(EXPERIMENT_DIR, "training_history.pkl")
if os.path.exists(history_pkl_path):
    mlflow.log_artifact(history_pkl_path, artifact_path="history")

# --- Training curves (matching 4.0 notebook) as MLflow artifacts ---
if history and (history.get("step") or history.get("epoch_step")):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Loss curves
    ax = axes[0]
    if history.get("step"):
        ax.plot(history["step"], history["train_loss"], 'b-', alpha=0.6, label='Train loss (checkpoint)')
        ax.plot(history["step"], history["val_loss"], 'r-', alpha=0.6, label='Val loss (checkpoint)')
    if history.get("epoch_step"):
        ax.plot(history["epoch_step"], history["epoch_val_loss"], 'ro-', markersize=5, label='Val loss (end-of-epoch)')
    ax.set_xlabel('Global Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: F1 curves
    ax = axes[1]
    if history.get("step"):
        ax.plot(history["step"], history["val_f1"], 'r-', alpha=0.6, label='Val F1 (checkpoint)')
    if history.get("epoch_step"):
        if history.get("epoch_train_f1"):
            ax.plot(history["epoch_step"], history["epoch_train_f1"], 'b^-', markersize=5, label='Train F1 (end-of-epoch)')
        ax.plot(history["epoch_step"], history["epoch_val_f1"], 'ro-', markersize=5, label='Val F1 (end-of-epoch)')
    ax.set_xlabel('Global Step')
    ax.set_ylabel('F1 Score')
    ax.set_title('Training & Validation F1')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle(f'GNN Training Curves — {FINAL_TAG}', fontsize=13)
    plt.tight_layout()

    # Save to experiment dir and log as MLflow artifact
    _curves_path = os.path.join(EXPERIMENT_DIR, "training_curves.png")
    fig.savefig(_curves_path, dpi=150, bbox_inches='tight')
    mlflow.log_artifact(_curves_path, artifact_path="plots")
    plt.close(fig)
    print(f"Logged training curves artifact: {_curves_path}")

mlflow.end_run()
print(f"\nMLflow run completed: {mlflow_run.info.run_id}")